- From eda I have found that the "car_age" will be helpfull in terms of classic cars
- Extracting the model and variant from name will also improve the model prediction

In [180]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime

In [181]:
path = '../data'

In [182]:
df = pd.read_csv(f"{path}/pakwheels_data.csv")
df.head()

,name,brand,model,mileage_km,fuel_type,transmission,engine_cc,city,price_pkr
0,Toyota Surf 1997 for Sale,Toyota,1997,100000,Petrol,Automatic,2700,Gujrat,4100000
1,Suzuki Jimny 1991 for Sale,Suzuki,1991,130000,Petrol,Manual,1000,Gujrat,1100000
2,Honda City 2004 i-DSI Vario for Sale,Honda,2004,250000,Petrol,Automatic,1300,Lahore,1825000
3,Honda City 2005 i-DSI for Sale,Honda,2005,250000,Petrol,Manual,1300,Lahore,1625000
4,Toyota IST 2003 1.3 A for Sale,Toyota,2003,100000,Petrol,Automatic,1300,Karachi,1500000


In [183]:
df = df.drop_duplicates()
df.duplicated().sum()

0

In [184]:
df.shape

(24731, 9)

In [185]:
df['car_age'] = int(datetime.now().year)-df['model']
df.rename(columns={'model': 'year'}, inplace=True)

In [186]:
df.head()

,name,brand,year,mileage_km,fuel_type,transmission,engine_cc,city,price_pkr,car_age
0,Toyota Surf 1997 for Sale,Toyota,1997,100000,Petrol,Automatic,2700,Gujrat,4100000,29
1,Suzuki Jimny 1991 for Sale,Suzuki,1991,130000,Petrol,Manual,1000,Gujrat,1100000,35
2,Honda City 2004 i-DSI Vario for Sale,Honda,2004,250000,Petrol,Automatic,1300,Lahore,1825000,22
3,Honda City 2005 i-DSI for Sale,Honda,2005,250000,Petrol,Manual,1300,Lahore,1625000,21
4,Toyota IST 2003 1.3 A for Sale,Toyota,2003,100000,Petrol,Automatic,1300,Karachi,1500000,23


In [187]:
import re

def parse_name(row):
    name = row['name']
    brand = row['brand']
    year = row['year']
    
    name = re.sub(r'\s*for sale\s*$', '', name, flags=re.I).strip()
    
    m = re.match(rf'^(.*?)\s*\b{year}\b\s*(.*)$', name)
    if not m: return pd.Series([None, None])
    
    left, variant = m.group(1).strip(), m.group(2).strip()
    
    car_model = re.sub(rf'^{re.escape(brand)}\s*','', left, flags=re.I).strip()
    
    return pd.Series([car_model or None, variant or None])

In [188]:
df[['model', 'variant']] = df.apply(parse_name, axis=1)
df.head()

,name,brand,year,mileage_km,fuel_type,transmission,engine_cc,city,price_pkr,car_age,model,variant
0,Toyota Surf 1997 for Sale,Toyota,1997,100000,Petrol,Automatic,2700,Gujrat,4100000,29,Surf,None
1,Suzuki Jimny 1991 for Sale,Suzuki,1991,130000,Petrol,Manual,1000,Gujrat,1100000,35,Jimny,None
2,Honda City 2004 i-DSI Vario for Sale,Honda,2004,250000,Petrol,Automatic,1300,Lahore,1825000,22,City,i-DSI Vario
3,Honda City 2005 i-DSI for Sale,Honda,2005,250000,Petrol,Manual,1300,Lahore,1625000,21,City,i-DSI
4,Toyota IST 2003 1.3 A for Sale,Toyota,2003,100000,Petrol,Automatic,1300,Karachi,1500000,23,IST,1.3 A


In [189]:
df.sample(30)[["name", "year", "model", "variant"]] 
(df["car_age"] != 2026 - df["year"]).sum() 

0

In [190]:
df[df['variant'].isna()].count()

name            3629
brand           3629
year            3629
mileage_km      3629
fuel_type       3629
transmission    3629
engine_cc       3629
city            3629
price_pkr       3629
car_age         3629
model           3629
variant            0
dtype: int64

In [191]:
df['variant'].isna().sum()

3629

In [192]:
df[df['variant'].isna()][['name', 'brand', 'year', 'variant']].head(10)

,name,brand,year,variant
0,Toyota Surf 1997 for Sale,Toyota,1997,None
1,Suzuki Jimny 1991 for Sale,Suzuki,1991,None
13,Honda Accord 1987 for Sale,Honda,1987,None
15,Suzuki Mehran 2007 for Sale,Suzuki,2007,None
21,Mitsubishi Pajero Mini 1999 for Sale,Mitsubishi,1999,None
22,Hyundai Excel 1993 for Sale,Hyundai,1993,None
30,Honda Civic 2006 for Sale,Honda,2006,None
36,KIA Sportage 2002 for Sale,KIA,2002,None
58,Nissan Juke 2011 for Sale,Nissan,2011,None
63,Toyota Surf 1997 for Sale,Toyota,1997,None


In [193]:
df[df['variant'].notna()][['name', 'brand', 'variant']].head(10)

,name,brand,variant
2,Honda City 2004 i-DSI Vario for Sale,Honda,i-DSI Vario
3,Honda City 2005 i-DSI for Sale,Honda,i-DSI
4,Toyota IST 2003 1.3 A for Sale,Toyota,1.3 A
5,Toyota Mark X 2005 250 G L Package for Sale,Toyota,250 G L Package
6,Toyota Land Cruiser 5th (J80) Generation 1995 ...,Toyota,VX Limited 4.5
7,Daihatsu Cuore 2006 CL for Sale,Daihatsu,CL
8,Toyota Prado 2011 TX 2.7 for Sale,Toyota,TX 2.7
9,Honda City 2005 i-DSI Vario for Sale,Honda,i-DSI Vario
10,Honda Civic 2005 EXi Prosmatec for Sale,Honda,EXi Prosmatec
11,Daihatsu Cuore 2002 CX Automatic for Sale,Daihatsu,CX Automatic


In [194]:
df.to_csv(f'{path}/feature_engineered_pakwheels_data.csv', index=False)

In [195]:
df = pd.read_csv(f'{path}/feature_engineered_pakwheels_data.csv')
df.head()

,name,brand,year,mileage_km,fuel_type,transmission,engine_cc,city,price_pkr,car_age,model,variant
0,Toyota Surf 1997 for Sale,Toyota,1997,100000,Petrol,Automatic,2700,Gujrat,4100000,29,Surf,NaN
1,Suzuki Jimny 1991 for Sale,Suzuki,1991,130000,Petrol,Manual,1000,Gujrat,1100000,35,Jimny,NaN
2,Honda City 2004 i-DSI Vario for Sale,Honda,2004,250000,Petrol,Automatic,1300,Lahore,1825000,22,City,i-DSI Vario
3,Honda City 2005 i-DSI for Sale,Honda,2005,250000,Petrol,Manual,1300,Lahore,1625000,21,City,i-DSI
4,Toyota IST 2003 1.3 A for Sale,Toyota,2003,100000,Petrol,Automatic,1300,Karachi,1500000,23,IST,1.3 A
